<a href="https://colab.research.google.com/github/Sakith-N/Statistical-Learning-e22252/blob/assignment--7/2D_Position_Estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part A.

$p_x(k) = p_x(k-1) + v_x(k-1)\Delta t + \frac{1}{2}\Delta t^2 w_x(k-1)$

$p_y(k) = p_y(k-1) + v_y(k-1)\Delta t + \frac{1}{2}\Delta t^2 w_y(k-1)$

$p_y(k) = p_y(k-1) + v_y(k-1)\Delta t + \frac{1}{2}\Delta t^2 w_y(k-1)$

$v_y(k) = v_y(k-1) + \Delta t w_y(k-1)$

Expressing these linear kinematic dependencies in matrix format,

$\begin{bmatrix} p_x(k) \\ p_y(k) \\ v_x(k) \\ v_y(k) \end{bmatrix} = \begin{bmatrix} 1 & 0 & \Delta t & 0 \\ 0 & 1 & 0 & \Delta t \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix} \begin{bmatrix} p_x(k-1) \\ p_y(k-1) \\ v_x(k-1) \\ v_y(k-1) \end{bmatrix} + \begin{bmatrix} \frac{1}{2}\Delta t^2 & 0 \\ 0 & \frac{1}{2}\Delta t^2 \\ \Delta t & 0 \\ 0 & \Delta t \end{bmatrix} \begin{bmatrix} w_x(k-1) \\ w_y(k-1) \end{bmatrix}$


This confirms the derivations for state transition matrix A and noise gain matrix G.Since the GPS sensor only reads out  $p_x(k)$ and $p_y(k)$:

$y_k = \begin{bmatrix} p_x^{meas}(k) \\ p_y^{meas}(k) \end{bmatrix} = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \end{bmatrix} \begin{bmatrix} p_x(k) \\ p_y(k) \\ v_x(k) \\ v_y(k) \end{bmatrix} + z_k$

In [ ]:
import numpy as np

class KF2D:
    def __init__(self, delta_t, sigma_p_variance, sigma_m_variance):
        self.dt = delta_t

        # Initialize Matrix Structures
        self.A = np.array([
            [1, 0, self.dt, 0],
            [0, 1, 0, self.dt],
            [0, 0, 1, 0],
            [0, 0, 0, 1]
        ])

        self.G = np.array([
            [0.5 * (self.dt**2), 0],
            [0, 0.5 * (self.dt**2)],
            [self.dt, 0],
            [0, self.dt]
        ])

        self.H = np.array([
            [1, 0, 0, 0],
            [0, 1, 0, 0]
        ])

        # Noise Covariance Configurations
        self.Sigma_p = np.eye(2) * sigma_p_variance
        self.Q = self.G @ self.Sigma_p @ self.G.T
        self.R = np.eye(2) * sigma_m_variance

        # Initial Assumptions (State vector & Covariance matrix)
        self.m = np.array([0.0, 0.0, 0.0, 0.0])
        self.P = np.eye(4) * 10.0

    def filter_sequence(self, measurements):
        filtered_positions = []

        for obs in measurements:
            Mminus = self.A @ self.m
            Pminus = self.A @ self.P @ self.A.T + self.Q

            S = self.H @ Pminus @ self.H.T + self.R
            K = Pminus @ self.H.T @ np.linalg.inv(S)

            self.m = Mminus + K @ (obs - self.H @ Mminus)
            self.P = (np.eye(4) - K @ self.H) @ Pminus

            filtered_positions.append(self.m[0:2].copy())

        return np.array(filtered_positions)

if __name__ == "__main__":
    gps_measurements = np.array([
        [1.1, 0.9],
        [2.0, 2.1],
        [2.9, 3.0],
        [4.2, 3.8]
    ])

    print("Part B.")
    kf = KF2D(delta_t=1.0, sigma_p_variance=0.1, sigma_m_variance=0.5)
    clean_trajectory = kf.filter_sequence(gps_measurements)
    print("Filtered Position Path Output:\n", clean_trajectory)

Part B.
Filtered Position Path Output:
 [[1.07320341 0.87807552]
 [1.97095082 2.04153705]
 [2.88608152 3.0234984 ]
 [4.0758389  3.87210176]]
